just need to pass the data through the trained i3d and get the features vectors

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

In [ ]:
TFRECORD_BASE = '/kaggle/input/datasets/ganuwoahh/cholec80-tfrecords-real/tfrecords'
MODEL_PATH = '/kaggle/input/models/ganuwoahh/140426-final-keras/pytorch/default/1/140426_final.keras'
OUTPUT_DIR = '/kaggle/working/extracted_features'

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
print("Loading fully trained I3D model...")
base_model = load_model(MODEL_PATH)

feature_layer = base_model.layers[-3].output

model = Model(inputs=base_model.inputs, outputs=feature_layer)
print(f"Model decapitated! It will now output raw {feature_layer.shape[-1]}-dimensional thought vectors.")

In [ ]:
def parse_tfrecord(example): # same function as training notebook
    feature_description = {
        'rgb_image': tf.io.FixedLenFeature([], tf.string),
        'flow_image': tf.io.FixedLenFeature([], tf.string),
        'label': tf.io.FixedLenFeature([], tf.int64),
    }
    parsed = tf.io.parse_single_example(example, feature_description)
    
    rgb = tf.image.decode_jpeg(parsed['rgb_image'], channels=3)
    rgb = tf.cast(rgb, tf.float32) / 127.5 - 1.0 
    
    flow = tf.cond(
        tf.equal(tf.strings.length(parsed['flow_image']), 0),
        lambda: tf.ones([224, 224, 3], dtype=tf.float32) * 128.0, 
        lambda: tf.cast(tf.image.decode_jpeg(parsed['flow_image'], channels=3), tf.float32)
    )
    flow = (flow[:, :, :2] - 128.0) / 128.0 
    
    return {'rgb': rgb, 'flow': flow}, parsed['label']

In [ ]:
def build_extraction_dataset(filepath, window_size=32, batch_size=8): # almost same function as training notebook just no interleave no shuffle because lstm needs sequential data
    ds = tf.data.TFRecordDataset(filepath, num_parallel_reads=1)
    ds = ds.map(parse_tfrecord, num_parallel_calls=tf.data.AUTOTUNE)
    
    ds = ds.window(size=window_size, shift=8, drop_remainder=True)
    ds = ds.flat_map(lambda x, y: tf.data.Dataset.zip((
        {
            'rgb': x['rgb'].batch(window_size), 
            'flow': x['flow'].batch(window_size)
        }, 
        y.batch(window_size)
    )))
    
    ds = ds.map(lambda inputs, labels: (
        (inputs['rgb'], inputs['flow']), 
        tf.one_hot(labels[-1], depth=7)
    ), num_parallel_calls=tf.data.AUTOTUNE)
    
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

In [ ]:
for vid_idx in tqdm(range(1, 81), desc="Extracting Videos"):
    vid_name = f'video{vid_idx:02d}'
    filepath = os.path.join(TFRECORD_BASE, f'{vid_name}.tfrecord')
    
    if not os.path.exists(filepath):
        continue
        
    dataset = build_extraction_dataset(filepath)
    
    video_features = []
    video_labels = []
    
    for (rgb, flow), labels in dataset:
        features = model.predict_on_batch([rgb, flow])
        
        video_features.append(features) # stacking them and saving the 2d matrix for that video as a features.npy and a labels.npy
        video_labels.append(labels.numpy())
        
    if video_features:
        video_features = np.concatenate(video_features, axis=0)
        video_labels = np.concatenate(video_labels, axis=0)
        
        np.save(os.path.join(OUTPUT_DIR, f'{vid_name}_features.npy'), video_features)
        np.save(os.path.join(OUTPUT_DIR, f'{vid_name}_labels.npy'), video_labels)

print(f"Vectors saved to {OUTPUT_DIR}")

In [ ]:
!zip -r /kaggle/working/cholec80_i3d_features.zip /kaggle/working/extracted_features